## ML_1045: Byte-Pair Encoding (BPE)

**Difficulty**: Hard | **TorchCode**: #35

### Problem
Implement BPE tokenizer training and encoding. Training: split words into characters + `</w>`, iteratively find and merge the most frequent adjacent pair. Encoding: apply learned merges in order to tokenize new text.

### Formula
$$\text{merge}^* = \arg\max_{(a,b)} \text{freq}(a, b), \quad \text{repeat } n \text{ times}$$

In [ ]:
class SimpleBPE:
    def __init__(self):
        self.merges = []

    def train(self, corpus, num_merges):
        vocab = {}
        for word in corpus:
            symbols = tuple(word) + ('</w>',)
            vocab[symbols] = vocab.get(symbols, 0) + 1
        self.merges = []
        for _ in range(num_merges):
            pairs = {}
            for word, freq in vocab.items():
                for i in range(len(word) - 1):
                    pair = (word[i], word[i + 1])
                    pairs[pair] = pairs.get(pair, 0) + freq
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            self.merges.append(best)
            new_vocab = {}
            for word, freq in vocab.items():
                new_word = []
                i = 0
                while i < len(word):
                    if i < len(word) - 1 and (word[i], word[i + 1]) == best:
                        new_word.append(word[i] + word[i + 1])
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_vocab[tuple(new_word)] = freq
            vocab = new_vocab

    def encode(self, text):
        all_tokens = []
        for word in text.split():
            symbols = list(word) + ['</w>']
            for a, b in self.merges:
                i = 0
                while i < len(symbols) - 1:
                    if symbols[i] == a and symbols[i + 1] == b:
                        symbols = symbols[:i] + [a + b] + symbols[i + 2:]
                    else:
                        i += 1
            all_tokens.extend(symbols)
        return all_tokens

In [ ]:
# --- Test 1: Correct number of merges ---
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=5)
assert len(bpe.merges) == 5

# --- Test 2: Most frequent pair merged first ---
bpe2 = SimpleBPE()
bpe2.train(['aaa', 'aaa', 'aaa', 'bbb'], num_merges=1)
assert bpe2.merges[0] == ('a', 'a')

# --- Test 3: Encode returns list of strings and reconstructs correctly ---
bpe3 = SimpleBPE()
bpe3.train(['low', 'lower', 'lowest'] * 3, num_merges=10)
tokens = bpe3.encode('low')
assert isinstance(tokens, list)
assert all(isinstance(t, str) for t in tokens)
reconstructed = ''.join(t.replace('</w>', '') for t in tokens)
assert reconstructed == 'low'

# --- Test 4: More merges → fewer tokens ---
bpe4 = SimpleBPE()
bpe4.train(['hello'] * 10, num_merges=2)
bpe5 = SimpleBPE()
bpe5.train(['hello'] * 10, num_merges=10)
assert len(bpe5.encode('hello')) <= len(bpe4.encode('hello'))

print('All tests passed!')